Testujemy Api od pse

In [ ]:
# Test pierwszy
r = requests.get("https://api.raporty.pse.pl/api/", headers=HEADERS, timeout=30)
print(r.status_code)
print(r.text[:3000])

# Test drugi
candidates = ["pdgopkd", "pdgobpkd"]

for ep in candidates:
    print(f"\n=== {ep} ===")
    for d in [date(2023, 1, 1), date(2023, 6, 1), date(2024, 1, 1), date(2024, 6, 1), date(2025, 1, 1)]:
        url = f"https://api.raporty.pse.pl/api/{ep}"
        params = {"$filter": f"business_date eq '{d.isoformat()}'"}
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=15)
            if r.status_code == 200:
                records = r.json().get("value", [])
                if records:
                    keys = list(records[0].keys())
                    has_oze = any("wi" in k or "fv" in k or "pv" in k for k in keys)
                    print(f"  {d}: {len(records)} rekordów, OZE: {has_oze}")
                    if d == date(2024, 6, 1):
                        print(f"     pola: {keys}")
                else:
                    print(f"  {d}: 0 rekordów")
            else:
                print(f"  {d}: status {r.status_code}")
        except Exception as e:
            print(f"  {d}: błąd {e}")

# Test trzeci
candidates = ["pk5l-wp", "prog-obc", "his-obc"]

for ep in candidates:
    print(f"\n=== {ep} ===")
    for d in [date(2023, 6, 1), date(2024, 6, 1), date(2025, 6, 1)]:
        url = f"https://api.raporty.pse.pl/api/{ep}"
        params = {"$filter": f"business_date eq '{d.isoformat()}'"}
        try:
            r = requests.get(url, params=params, headers=HEADERS, timeout=15)
            if r.status_code == 200:
                records = r.json().get("value", [])
                print(f"  {d}: {len(records)} rekordów", end="")
                if records:
                    keys = list(records[0].keys())
                    has_oze = any(k in str(keys).lower() for k in ["wi", "fv", "pv", "wiatr", "foto"])
                    print(f", OZE: {has_oze}")
                    if records and d == date(2023, 6, 1):
                        print(f"    pola: {keys}")
                else:
                    print()
            else:
                print(f"  {d}: status {r.status_code}")
        except Exception as e:
            print(f"  {d}: błąd {e}")


# Test dat
test_dates = [
    date(2023, 1, 1),
    date(2023, 6, 1),
    date(2024, 1, 1),
    date(2024, 6, 1),
    date(2024, 7, 1),
    date(2025, 1, 1),
]

for d in test_dates:
    params = {"$filter": f"business_date eq '{d.isoformat()}'"}
    try:
        r = requests.get(PSE_API_URL, params=params, headers=HEADERS, timeout=30)
        n = len(r.json().get("value", []))
        print(f"{d}: {n} rekordów")
    except Exception as e:
        print(f"{d}: błąd - {e}")